# RIAWELC Weld Defect Classification — Training

Runs training experiments and logs to W&B.

**Before running:**
1. Add your W&B API key as a Kaggle Secret named `WANDB_API_KEY`
2. Attach the RIAWELC dataset to this notebook

In [ ]:
# --- W&B login via Kaggle Secret ---
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

In [ ]:
# --- Install dependencies ---
!pip install -q lightning torchmetrics wandb grad-cam

In [ ]:
# --- Clone repo ---
!git clone https://github.com/Alexanderfrs/weld-defect-classification-RIAWELC-deeplify.git repo
%cd repo

In [ ]:
# --- Find dataset path ---
!find /kaggle/input -name "NoDifetto" -type d

In [ ]:
# --- Set DATA_ROOT ---
# Set this to the folder that contains training/ validation/ testing/ subfolders.
# Use the path from the find output above, going up two levels from any NoDifetto folder.
import os
os.environ["RIAWELC_DATA_ROOT"] = "/kaggle/input/datasets/alexanderfries/riawelc-dataset/DB - Copy"

In [ ]:
# --- Verify clean split (no train/test overlap) ---
import sys
sys.path.insert(0, '.')
from src.splits import build_clean_splits
tr, va, te = build_clean_splits()
tr_f = {(p.parent.name, p.name) for p, _ in tr}
te_f = {(p.parent.name, p.name) for p, _ in te}
overlap = len(tr_f & te_f)
print(f'Train/Test overlap: {overlap} (must be 0)')
assert overlap == 0, 'ERROR: overlap found!'
print('OK — split is clean')

In [ ]:
# --- Run 2: Finetune (unfrozen backbone, CE loss, 20 epochs) ---
!python -m src.train run_2_finetune

In [ ]:
# --- Optional: Run 3 (Focal Loss) ---
# !python -m src.train run_3_focal

In [ ]:
# --- Optional: Run 1 (baseline frozen) ---
# !python -m src.train run_1_baseline